# Databricks Notebook — Auto-Generated

**Status:** Packaged under fail-safe (max revisions reached)

---

## Execution Plan

Step 1: Environment Setup [CODE]
  - Import necessary libraries: pyspark.sql.functions, pyspark.sql.types, and requests for webhook alerts
  - Set spark.sql.shuffle.partitions to auto for optimized performance
  - Retrieve S3 access keys and webhook URL from Databricks secrets using dbutils.secrets.get() from 'prod-scope'

Step 2: Data Ingestion [CODE]
  - Read customer records from the S3 bucket s3://my-company/customers/raw/ using spark.read.json()
  - Print the schema and row count of the ingested data for validation

Step 3: Data Cleaning [CODE]
  - Clean the email column by stripping whitespace, converting to lowercase, and dropping rows with null emails
  - Validate the cleaning process by showing a sample of the cleaned data

Step 4: Add Audit Column [CODE]
  - Add a new column 'ingestion_timestamp' to the DataFrame using current_timestamp() function
  - Display a sample of the data to verify the new column addition

Step 5: Load / Merge into Delta [CODE]
  - Use Delta Lake MERGE INTO to upsert records into the Delta table at dbfs:/delta/customers/
  - Match records on the primary key 'customer_id' to ensure no destructive overwrites occur

Step 6: Row Count Validation [CODE]
  - Calculate the row count of the DataFrame after cleaning
  - If the row count is less than 1000, trigger an alert

Step 7: Send Alert via Webhook [CODE]
  - If the row count is below 1000, send a POST request to the webhook URL retrieved from secrets
  - Include a message in the alert indicating the row count drop below the threshold

Step 8: Error Handling and Logging [CODE]
  - Implement try-except blocks around critical operations to catch and log errors
  - Log errors and alerts to a monitoring system or file for further analysis

---

*Generated by Databricks Notebook Generator · 2026-03-08 18:51 UTC*

In [ ]:
# Step 1: Environment Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import requests
import logging

# Initialize Spark session
spark = SparkSession.builder.appName("CustomerDataPipeline").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "auto")

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Retrieve secrets
s3_access_key = dbutils.secrets.get(scope="prod-scope", key="s3-access-key")
s3_secret_key = dbutils.secrets.get(scope="prod-scope", key="s3-secret-key")
webhook_url = dbutils.secrets.get(scope="prod-scope", key="webhook-url")

In [ ]:
# Step 2: Data Ingestion
try:
    schema = StructType([
        StructField("customer_id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("email", StringType(), True)
    ])
    
    customer_df = spark.read.schema(schema).json("s3a://my-company/customers/raw/")
    logger.info("Data ingestion completed successfully.")
    display(customer_df)
except Exception as e:
    logger.error(f"Error during data ingestion: {e}")

In [ ]:
# Step 3: Data Cleaning
try:
    cleaned_df = customer_df.withColumn("email", lower(trim(col("email")))).dropna(subset=["email"])
    logger.info("Data cleaning completed successfully.")
    display(cleaned_df.limit(10))
except Exception as e:
    logger.error(f"Error during data cleaning: {e}")

In [ ]:
# Step 4: Add Audit Column
try:
    enriched_df = cleaned_df.withColumn("ingestion_timestamp", current_timestamp())
    logger.info("Audit column added successfully.")
    display(enriched_df.limit(10))
except Exception as e:
    logger.error(f"Error adding audit column: {e}")

In [ ]:
# Step 5: Load / Merge into Delta
try:
    enriched_df.createOrReplaceTempView("updates")

    spark.sql("""
        MERGE INTO delta.`dbfs:/delta/customers/` AS target
        USING updates AS source
        ON target.customer_id = source.customer_id
        WHEN MATCHED THEN
          UPDATE SET *
        WHEN NOT MATCHED THEN
          INSERT *
    """)
    logger.info("Data merged into Delta table successfully.")
except Exception as e:
    logger.error(f"Error during Delta Lake merge: {e}")

In [ ]:
# Step 6: Row Count Validation
try:
    row_count = enriched_df.count()
    if row_count < 1000:
        alert_message = f"Row count below threshold: {row_count}"
        logger.warning(alert_message)
except Exception as e:
    logger.error(f"Error during row count validation: {e}")

In [ ]:
# Step 7: Send Alert via Webhook
try:
    if row_count < 1000:
        response = requests.post(webhook_url, json={"text": alert_message})
        if response.status_code != 200:
            logger.error(f"Failed to send alert: {response.text}")
except Exception as e:
    logger.error(f"Error sending alert: {e}")

In [ ]:
# Step 8: Error Handling and Logging
# (Already implemented try-except blocks with logging in each step above)